In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.silver_fuel_purchases"
    )
)

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.silver_trips"
    )
)

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.silver_routes"
    )
)

In [0]:
from pyspark.sql import functions as F

silver_fuel_purchases = spark.table(
    "workspace.transportation_analytics.silver_fuel_purchases"
)

silver_trips = spark.table(
    "workspace.transportation_analytics.silver_trips"
)

silver_routes = spark.table(
    "workspace.transportation_analytics.silver_routes"
)

print("Silver tables loaded successfully")

In [0]:
gold_fuel_efficiency = (
    silver_trips
    .groupBy("dispatch_date")
    .agg(
        F.avg("average_mpg").alias("avg_mpg"),
        F.sum("fuel_gallons_used").alias("total_fuel_gallons"),
        F.sum("actual_distance_miles").alias("total_distance_miles")
    )
    .orderBy("dispatch_date")
)

display(gold_fuel_efficiency)

In [0]:
gold_fuel_efficiency = (
    silver_trips
    .join(
        fuel_by_trip,
        on="trip_id",
        how="left"
    )
    .join(
        silver_loads.select(
            "load_id",
            "route_id"
        ),
        on="load_id",
        how="left"
    )
    .join(
        silver_routes.select(
            "route_id",
            "origin_city",
            "destination_city"
        ),
        on="route_id",
        how="left"
    )
    .groupBy(
        "dispatch_date",
        "route_id",
        "origin_city",
        "destination_city"
    )
    .agg(
        F.avg("average_mpg").alias("avg_mpg"),
        F.sum("trip_fuel_cost").alias("total_fuel_cost"),
        F.sum("fuel_gallons_used").alias("total_fuel_gallons"),
        F.sum("actual_distance_miles").alias("total_distance_miles")
    )
    .orderBy("dispatch_date")
)

display(gold_fuel_efficiency)

In [0]:
gold_fuel_efficiency.limit(0).write \
    .format("delta") \
    .saveAsTable(
        "workspace.transportation_analytics.gold_fuel_efficiency"
    )

print("Empty Gold Fuel Efficiency table created successfully")

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_fuel_efficiency"
)

target.alias("t").merge(
    gold_fuel_efficiency.alias("s"),
    """
    t.dispatch_date = s.dispatch_date
    AND t.route_id = s.route_id
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Fuel Efficiency table updated using MERGE")

In [0]:
display(
    spark.table(
        "workspace.transportation_analytics.gold_fuel_efficiency"
    )
)

In [0]:
display(
    gold_fuel_efficiency
    .orderBy("dispatch_date")
    .select(
        "dispatch_date",
        "avg_mpg"
    )
)

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    gold_fuel_efficiency
    .groupBy(
        "route_id",
        "origin_city",
        "destination_city"
    )
    .agg(
        F.sum("total_fuel_cost").alias("route_fuel_cost")
    )
    .orderBy(F.desc("route_fuel_cost"))
)

Databricks visualization. Run in Databricks to view.